Prerequisites:

- `pod5` files from sequencer in folder `pod5s`
- `fastq` files from sequencer in separate subfolders per sample in folder `sequencer fastq`
- A reference genome from NCBI (GCF_000006765.1) (`ncbi_reference.fna`)

## Setting some paths

In [ ]:
modkit="/home/u0165050/bin/modkit/modkit"
dorado="home/u0165050/bin/Dorado/bin/dorado"
dorado_models="home/u0165050/bin/Dorado/models"

## Making reference assembly (empty_not_induced)

### Read QC filtering

QC filtering was done for each Nanopore read set using `fastp`.

In [ ]:
dir -1 sequencer_fastqs | xargs -I % fastp \
-i sequencer_fastqs/%/long_reads.fastq.gz \
-o sequencer_fastqs/%/long_reads_filtered.fastq.gz \
-e 18 -5 18

### Assembly

Make a reference-based assembly using using `rebaler` with the reads of the empty-not-induced sample, and the NCBI reference genome.

In [ ]:
mkdir -p assemblies
conda activate rebaler

In [ ]:
rebaler ncbi_reference.fna sequencer_fastqs/empty_not_induced/long_reads_filtered.fastq.gz \
> assemblies/empty_not_induced.fasta

In [ ]:
conda deactivate

## Modified basecalling

In [ ]:
mkdir -p raw_methylation_bams
dir -1 pod5s | xargs basename -s .pod5 | xargs -I % bash -c \
"$dorado basecaller sup,6mA,4mC_5mC pod5s/%/ -r \
-x cuda:all \
--models-directory $dorado_models \
> raw_basecalled_bams/%.bam || touch %.failed"

## Mapping, sorting and indexing bams

`minimap2`'s option `-y` is crucial: it copies the comments from the `fastq` file (i.e. the methylation information) to the alignments.

In [ ]:
mkdir -p sorted_bams
dir -1 raw_basecalled_bams | xargs basename -s .bam | xargs -I % bash -c \
'samtools fastq raw_methylation_bams/%.bam -T MM,ML -@ 22 | \
minimap2 -t 22 --secondary=no -ax map-ont -y assemblies/empty_not_induced.fasta - | \
samtools view -b -@ 22 | \
samtools sort -@ 22 -o sorted_bams/%.bam && \
samtools index -@ 22 sorted_bams/%.bam'

## Making a pile-up of the reads using `modkit`

In [ ]:
mkdir -p bedMethyls
dir -1 sorted_bams | grep -E '\.bam$' | xargs basename -s .bam | xargs -I % \
$modkit pileup --threads 22 sorted_bams/%.bam bedMethyls/%.bed --ref assemblies/empty_not_induced.fasta --log-filepath bedMethyls/%.log \
2>/dev/null

Compressing and indexing

In [ ]:
cd bedMethyls
dir -1 | xargs -I % bgzip %
dir -1 | xargs -I % tabix %
cd ..

## Compute global methylation fractions

Get reference genome length

In [ ]:
samtools faidx assemblies/empty_not_induced.fasta
genome_length=$(cut -f 2 assemblies/empty_not_induced.fasta.fai)

In [ ]:
dir -1 bedMethyls | grep -E '\.bed\.gz$' | xargs -P 4 -I % python global_methylation.py bedMethyls/% $genome_length 50